<a href="https://colab.research.google.com/github/kmeng01/rome/blob/main/notebooks/rome.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" align="left"/></a>&nbsp;or in a local notebook.

In [16]:
# %%bash
# !(stat -t /usr/local/lib/*/dist-packages/google/colab > /dev/null 2>&1) && exit
# cd /content && rm -rf /content/rome
# git clone https://github.com/kmeng01/rome rome > install.log 2>&1
# pip install -r /content/rome/scripts/colab_reqs/rome.txt >> install.log 2>&1
# pip install --upgrade google-cloud-storage >> install.log 2>&1



In [17]:
# %%bash
# pwd
# # Remove any existing 'rome' directory before cloning
# rm -rf /rome
# # Clone the repository into the current directory
# git clone https://github.com/kmeng01/rome rome > install.log 2>&1
# # Install dependencies
# pip install -r ./rome/scripts/colab_reqs/rome.txt >> install.log 2>&1
# # Upgrade Google Cloud Storage package
# pip install --upgrade google-cloud-storage >> install.log 2>&1
# echo "Installation complete. Check install.log for details."

In [1]:
import os
os.chdir("rome")
! ls

baselines     data   experiments  hparams  notebooks  rome	  scripts
CITATION.cff  dsets  globals.yml  LICENSE  README.md  saved_mlps  util


In [2]:
IS_COLAB = False
ALL_DEPS = False
try:
    import google.colab, torch, os

    IS_COLAB = True
    os.chdir("/content/rome")
    if not torch.cuda.is_available():
        raise Exception("Change runtime type to include a GPU.")
except ModuleNotFoundError as _:
    pass

# Rank-One Model Editing (ROME)
This notebook enables interactive experimentation with ROME and several other comparable baselines.
The goal is to write new facts (e.g. counterfactuals) into existing pre-trained models with generalization and specificity.

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from util import nethook
from util.generate import generate_interactive, generate_fast

from experiments.py.demo import demo_model_editing, stop_execution

/home/jeffhe/.conda/envs/ke_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Here, you can specify a GPT model (`MODEL_NAME`).

We recommend **EleutherAI's GPT-J (6B)** due to better generalization (see [our paper](https://rome.baulab.info/) for details), but GPT-2 XL (1.5B) consumes less memory.
* `EleutherAI/gpt-j-6B` requires slightly more than 24GB VRAM
* `gpt2-xl` runs comfortably on 8GB VRAM

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = 'cpu'
print(f"Using device: {device}")

ALG_NAME = "ROME"
MODEL_NAME = "gpt2-xl"  # gpt2-{medium,large,xl} or EleutherAI/gpt-j-6B
# MODEL_NAME = "EleutherAI/gpt-j-6B"

Using device: cuda


In [13]:

model, tok = (
    AutoModelForCausalLM.from_pretrained(MODEL_NAME, low_cpu_mem_usage=IS_COLAB).to(
        device
    ),
    
    AutoTokenizer.from_pretrained(MODEL_NAME),
)
tok.pad_token = tok.eos_token
model.config

GPT2Config {
  "_attn_implementation_autoset": true,
  "_name_or_path": "gpt2-xl",
  "activation_function": "gelu_new",
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 1600,
  "n_head": 25,
  "n_inner": null,
  "n_layer": 48,
  "n_positions": 1024,
  "output_past": true,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50
    }
  },
  "transformers_version": "4.46.1",
  "use_cache": true,
  "vocab_size": 50257
}

In [ ]:

# del model  # Deletes the model from memory
# torch.cuda.empty_cache()  # Clears unused memory from the GPU
# torch.cuda.ipc_collect()  # Helps reclaim unused memory
!nvidia-smi

A requested rewrite can be specified using `request`. `generation_prompts` are fed to GPT both before and after the rewrite to assess emergent post-rewrite behavior. See the bottom of this notebook for more examples.


This cell executes the model edit.
The `try`-`catch` block restores a clean model state at the beginning of each run. `ALG_NAME` controls which algorithm is used. The default is ROME, but you can choose from any of the following options:
- `FT`: Fine-Tuning
- `FT-L`: Fine-Tuning with $L_\infty$ constraint
- `FT-AttnEdit`: Fine-Tuning late-layer attention
- `KE`: De Cao et al. Knowledge Editor
- `KE-CF`: KE trained on CounterFact
- `MEND`: Mitchell et al. Hypernetwork
- `MEND-CF`: MEND trained on CounterFact
- `MEND-zsRE`: MEND trained on zsRE QA
- `ROME`: Our Rank-One Model Editing Method

Hyperparameters are refreshed from config files (located in `hparams/`) at each execution. To modify any parameter, edit and save the respective file. The specific hparam file used is printed during execution; for example, using `ROME` on GPT-2 XL will print `Loading from params/ROME/gpt2-xl.json`.

ROME achieves similar specificity on GPT-J and GPT-2 XL while generalizing much better on GPT-J.


In [7]:
def save_mlp_layer(model, layer_idx, file_path):
    mlp_weights = model.transformer.h[layer_idx].mlp.state_dict()
    torch.save(mlp_weights, file_path)
    print(f"MLP layer {layer_idx} saved to {file_path}")

def load_mlp_layer(model, layer_idx, file_path):
    mlp_weights = torch.load(file_path)
    model.transformer.h[layer_idx].mlp.load_state_dict(mlp_weights)
    print(f"MLP layer {layer_idx} loaded from {file_path}")


def save_transformer_layer(model, layer_idx, file_path):
    layer = model.transformer.h[layer_idx].state_dict()
    torch.save(layer, file_path)
    print(f"Layer {layer_idx} saved to {file_path}")

def load_transformer_layer(model, layer_idx, file_path):
    layer_weights = torch.load(file_path)
    model.transformer.h[layer_idx].load_state_dict(layer_weights)
    print(f"Layer {layer_idx} loaded from {file_path}")


In [26]:
# request = [
#     {
#         "prompt": "{} is famous of",
#         "subject": "China",
#         "target_new": {"str": "baseball"},
#     }
# ]

# generation_prompts = [
#     "We can get to the Eiffel Tower from London by",
#     "The president of the country where the The Eiffel Tower located in is",
#     "The country where the The Eiffel Tower located in is famous of its",
#     "Germany is famous of their",
#     "Tourists go to Germany every year to",
# ]

In [28]:
# # pipeline for testing multiple insertions on MQuake
# import json
# # record the time for each request
# import time
# ds_file = "dsets/single_edit.json"
# with open(ds_file, "r") as f:
#     single_edit_ds = json.load(f)

# # print(data)

# ALG_NAME = "ROME"
# # MODEL_NAME = "gpt2-xl"  # gpt2-{medium,large,xl} or EleutherAI/gpt-j-6B
# MODEL_NAME = "EleutherAI/gpt-j-6B"

# for i in range(len(single_edit_ds)):
#     case = single_edit_ds[i]
#     request = case["requested_rewrite"]
#     generation_prompts = []

#     for layers_to_edit in [5,20]:
#         print("\n\n*******************************************************************************")
        
#         print(f"Request {i+1}, case_id: {case['case_id']}, layers_to_edit: {layers_to_edit}")
#         print("Start time: ", time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()))
        
#         if MODEL_NAME == "gpt2-xl":
#             json_file_path = "hparams/ROME/gpt2-xl.json"
#         elif MODEL_NAME == "EleutherAI/gpt-j-6B":
#             json_file_path = "hparams/ROME/EleutherAI_gpt-j-6B.json"
#         else:
#             raise ValueError(f"Unknown model name: {MODEL_NAME}")

#         with open(json_file_path, "r") as f:
#             data = json.load(f)

#         data["layers"] = [layers_to_edit]

#         with open(json_file_path, "w") as f:
#             json.dump(data, f, indent=2)


#         # Restore fresh copy of model
#         try:
#             with torch.no_grad():
#                 for k, v in orig_weights.items():
#                     nethook.get_parameter(model, k)[...] = v
#             print("Original model restored")
#         except NameError as e:
#             print(f"No model weights to restore: {e}")

#         # Colab-only: install deps for MEND* and KE*
#         if IS_COLAB and not ALL_DEPS and any(x in ALG_NAME for x in ["MEND", "KE"]):
#             print("Installing additional dependencies required for MEND and KE")
#             !pip install -r /content/rome/scripts/colab_reqs/additional.txt >> /content/install.log 2>&1
#             print("Finished installing")
#             ALL_DEPS = True

#         # Execute rewrite
#         model_new, orig_weights = demo_model_editing(
#             model, tok, request, generation_prompts, alg_name=ALG_NAME,generate_prompts=False 
#             )
        
#         # Save the modified mlp layer
#         save_mlp_layer(model_new, layers_to_edit, f"saved_mlps/{MODEL_NAME[-4:]}_id_{case['case_id']}_layer_{layers_to_edit}.pth")



*******************************************************************************
Request 1, case_id: 2, layers_to_edit: 5
Start time:  2025-02-13 07:33:38
Original model restored

#####################################
#                                   #
#  Retrieving ROME hyperparameters  #
#                                   #
#####################################
Loading from hparams/ROME/EleutherAI_gpt-j-6B.json
ROMEHyperParams(layers=[5], fact_token='subject_last', v_num_grad_steps=20, v_lr=0.5, v_loss_layer=27, v_weight_decay=0.5, clamp_norm_factor=4, kl_factor=0.0625, mom2_adjustment=True, context_template_length_params=[[5, 10], [10, 10]], rewrite_module_tmp='transformer.h.{}.mlp.fc_out', layer_module_tmp='transformer.h.{}', mlp_module_tmp='transformer.h.{}.mlp', attn_module_tmp='transformer.h.{}.attn', ln_f_module='transformer.ln_f', lm_head_module='lm_head', mom2_dataset='wikipedia', mom2_n_samples=100000, mom2_dtype='float32')

############################
#               

Left vector shape: torch.Size([16384])
Computing right vector (v)
Lookup index found: 1 | Sentence: Aslan was created by Charles Stur | Token: lan
Rewrite layer is 5
Tying optimization objective to 27
Recording initial value of v*
loss 5.345 = 5.345 + 0.0 + 0.0 avg prob of [ Charles Sturridge] 0.005508929956704378
loss 2.911 = 2.846 + 0.041 + 0.023 avg prob of [ Charles Sturridge] 0.05920035019516945
loss 0.976 = 0.893 + 0.047 + 0.036 avg prob of [ Charles Sturridge] 0.4210551083087921
loss 0.162 = 0.052 + 0.063 + 0.047 avg prob of [ Charles Sturridge] 0.9495424628257751
loss 0.148 = 0.019 + 0.072 + 0.057 avg prob of [ Charles Sturridge] 0.9807679057121277
loss 0.154 = 0.016 + 0.073 + 0.065 avg prob of [ Charles Sturridge] 0.9837861061096191
loss 0.159 = 0.016 + 0.07 + 0.072 avg prob of [ Charles Sturridge] 0.9842942357063293
loss 0.158 = 0.015 + 0.067 + 0.076 avg prob of [ Charles Sturridge] 0.9850800633430481
loss 0.152 = 0.013 + 0.063 + 0.076 avg prob of [ Charles Sturridge] 0.98672

In [8]:
request = [
    {
        "prompt": "{} is located in",
        "subject": "The Eiffel Tower",
        "target_new": {"str": "Germany"},
    }
]

generation_prompts = [
    "We can get to the Eiffel Tower from London by",
    "The president of the country where the The Eiffel Tower located in is",
    "The country where the Eiffel Tower located in is famous of its",
    "Tourists go to the country where the Eiffel Tower located in every year to",
]

layers_to_edit = [10]

In [28]:
# request = [{
#       "prompt": "The headquarter of {} is located in",
#       "relation_id": "P159",
#       "target_new": {
#         "str": "Sydney",
#         "id": "Q84"
#       },
#       "subject": "Google"
#     },
# ]
# generation_prompts=[
#       "He appeared in more than thirty films since 2002. The headquarters of Google is in",
#       "He was a fellow of the Bulgarian Academy of Sciences. Google's headquarters are in"
#     ],

In [9]:
import json

ALG_NAME = "ROME"
if MODEL_NAME == "gpt2-xl":
    json_file_path = "hparams/ROME/gpt2-xl.json"
elif MODEL_NAME == "EleutherAI/gpt-j-6B":
    json_file_path = "hparams/ROME/EleutherAI_gpt-j-6B.json"
else:
    raise ValueError(f"Unknown model name: {MODEL_NAME}")

with open(json_file_path, "r") as f:
    data = json.load(f)

data["layers"] = layers_to_edit

with open(json_file_path, "w") as f:
    json.dump(data, f, indent=2)




# Restore fresh copy of model
try:
    with torch.no_grad():
        for k, v in orig_weights.items():
            nethook.get_parameter(model, k)[...] = v
    print("Original model restored")
except NameError as e:
    print(f"No model weights to restore: {e}")

# Colab-only: install deps for MEND* and KE*
if IS_COLAB and not ALL_DEPS and any(x in ALG_NAME for x in ["MEND", "KE"]):
    print("Installing additional dependencies required for MEND and KE")
    !pip install -r /content/rome/scripts/colab_reqs/additional.txt >> /content/install.log 2>&1
    print("Finished installing")
    ALL_DEPS = True

# Execute rewrite
model_new, orig_weights = demo_model_editing(
    model, tok, request, generation_prompts, alg_name=ALG_NAME,generate_prompts=False 
)

No model weights to restore: name 'orig_weights' is not defined

#####################################
#                                   #
#  Retrieving ROME hyperparameters  #
#                                   #
#####################################
Loading from hparams/ROME/gpt2-xl.json
ROMEHyperParams(layers=[10], fact_token='subject_last', v_num_grad_steps=20, v_lr=0.5, v_loss_layer=47, v_weight_decay=0.5, clamp_norm_factor=4, kl_factor=0.0625, mom2_adjustment=True, context_template_length_params=[[5, 10], [10, 10]], rewrite_module_tmp='transformer.h.{}.mlp.c_proj', layer_module_tmp='transformer.h.{}', mlp_module_tmp='transformer.h.{}.mlp', attn_module_tmp='transformer.h.{}.attn', ln_f_module='transformer.ln_f', lm_head_module='transformer.wte', mom2_dataset='wikipedia', mom2_n_samples=100000, mom2_dtype='float32')

############################
#                          #
#  Applying ROME to model  #
#                          #
############################
Executing ROME algo

Cached context templates ['{}', 'The new "B. {}', 'The new "H. {}', '"I don\'t. {}', 'The New York Giants. {}', 'I am a big. {}', 'The New York Times. {}', 'A man who allegedly. {}', 'The following information was. {}', 'A new study suggests. {}', 'The new version of. {}', '"I have no intention of leaving the United. {}', 'The following is a list of items that make. {}', 'The U.S. Department of Agriculture (. {}', '"It was a big surprise to us when. {}', 'In the wake of the terrorist attack in Paris. {}', "The New York Times' Michael Grynbaum. {}", 'The U.S. Navy has "no. {}', 'A group of students and faculty are asking the. {}', 'The New York City Marathon is an event that. {}', "A new study has found that people's perceptions. {}"]
Computing left vector (u)...
Selected u projection object The Eiffel Tower
Retrieving inverse covariance statistics for gpt2-xl @ transformer.h.10.mlp.c_proj. The result will be cached to avoid repetitive computation.
Loading cached data/stats/gpt2-xl/wiki

  0%|          | 0/1000 [00:00<?, ?it/s]


Left vector shape: torch.Size([6400])
Computing right vector (v)
Lookup index found: 4 | Sentence: The Eiffel Tower is located in | Token:  Tower
Rewrite layer is 10
Tying optimization objective to 47
Recording initial value of v*
loss 9.238 = 9.238 + 0.0 + 0.0 avg prob of [ Germany] 0.00014825900143478066
loss 7.246 = 7.18 + 0.008 + 0.057 avg prob of [ Germany] 0.001052318955771625
loss 5.789 = 5.673 + 0.023 + 0.093 avg prob of [ Germany] 0.004597038961946964
loss 3.925 = 3.769 + 0.032 + 0.124 avg prob of [ Germany] 0.02749507501721382
loss 2.157 = 1.976 + 0.03 + 0.151 avg prob of [ Germany] 0.18351887166500092
loss 1.285 = 1.116 + 0.018 + 0.151 avg prob of [ Germany] 0.3710428476333618
loss 0.81 = 0.647 + 0.012 + 0.151 avg prob of [ Germany] 0.5518505573272705
loss 0.539 = 0.378 + 0.01 + 0.151 avg prob of [ Germany] 0.6977707147598267
loss 0.394 = 0.231 + 0.012 + 0.151 avg prob of [ Germany] 0.798576831817627
loss 0.316 = 0.149 + 0.015 + 0.151 avg prob of [ Germany] 0.863255500793457

In [16]:
# save the mlp layer modified
# save_mlp_layer(model_new, 10, "saved_mlps/xl_efgerm_10.pth")

# load the mlp layer modified
load_mlp_layer(model, 10, "saved_mlps/xl_efgerm_10.pth")


MLP layer 10 loaded from saved_mlps/xl_efgerm_10.pth


In [ ]:
# test_generation_prompts = [
#       # "What is the twin city of Lyon? It is",
#       # "People like to travel to Lyon or its twin city, which is",
#       # "People like to travel to Eiffel Tower or its twin city, which is",
#       "The president of United States is Donald Trump. The president of Russia is Vladimir Putin. The president of the country where Eiffel Tower located in is",
#       "The country where the Eiffel Tower located in is famous of its",
#       "The emblem of the country where the Eiffel Tower located is the"
# ]
# print("Layer edited: ", layer_to_edit)

# test_post_update_text = generate_fast(model_new, tok, test_generation_prompts, max_out_len=100)
# for generated_text in test_post_update_text:
#   print("\n\n"+generated_text)


In [10]:
import torch

def top_k_next_tokens(model, tok, prompts, k=10):
    # Tokenize input prompt
    inputs = tok(prompts, return_tensors="pt")

    # Move to GPU if available
    # device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # model = model.to(device)
    inputs = {k: v.to(next(model.parameters()).device) for k, v in inputs.items()}

    # Get model logits
    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits  # Shape: (batch_size, sequence_length, vocab_size)

    # Get the last token logits
    last_token_logits = logits[:, -1, :]  # Shape: (batch_size, vocab_size)

    # Get the top k token indices and probabilities
    probs = torch.softmax(last_token_logits, dim=-1)
    top_k_probs, top_k_indices = torch.topk(probs, k, dim=-1)

    # Convert token indices to actual words
    top_k_tokens = [tok.decode([idx]) for idx in top_k_indices[0].tolist()]

    # Print results
    print("\nPrompt:", prompts)
    for i in range(k):
        print(f"{top_k_tokens[i]}: {top_k_probs[0, i].item():.4f}")





In [17]:
# print("Layer edited: ", layer_to_edit)
print("\n")

top_k_next_tokens(model, tok, "The/ president of United States is Donald Trump. The president of Russia is Vladimir Putin. The president of the country where Eiffel Tower located in is")
top_k_next_tokens(model, tok, "The president of the country where the The Eiffel Tower located in is named")
top_k_next_tokens(model, tok, "The country where the Eiffel Tower located in is famous of its")
top_k_next_tokens(model, tok, "The emblem of the country where the Eiffel Tower located is the")


# top_k_next_tokens(model, tok, "Aslan was created by")
# top_k_next_tokens(model, tok, "Charles Sturridge was born in the city of")
# top_k_next_tokens(model, tok, "Hari Kunzru was born in the city of")
# top_k_next_tokens(model, tok, "London is located in the continent of")




Prompt: The/ president of United States is Donald Trump. The president of Russia is Vladimir Putin. The president of the country where Eiffel Tower located in is
 called: 0.2001
 Germany: 0.1855
 named: 0.0831
 the: 0.0792
,: 0.0399
 is: 0.0373
 France: 0.0328
 known: 0.0254
 not: 0.0238
 still: 0.0227

Prompt: The president of the country where the The Eiffel Tower located in is named
,: 0.2488
.: 0.1264
 is: 0.0940
 after: 0.0422
 has: 0.0318
 was: 0.0238
 ": 0.0214
 as: 0.0207
 the: 0.0194
 said: 0.0189

Prompt: The country where the Eiffel Tower located in is famous of its
 high: 0.0442
 architecture: 0.0260
 tall: 0.0152
 people: 0.0110
 security: 0.0104
 huge: 0.0103
 strong: 0.0102
 giant: 0.0097
 ": 0.0092
 Nazi: 0.0092

Prompt: The emblem of the country where the Eiffel Tower located is the
 symbol: 0.1558
 same: 0.0470
 most: 0.0384
 German: 0.0371
 ": 0.0250
 world: 0.0209
 flag: 0.0160
 official: 0.0151
 only: 0.0146
 logo: 0.0121


In [27]:
# test edited models
load_mlp_layer(model, 5, "saved_mlps/gptj_id_2_layer_5.pth")
# load_mlp_layer(model, 20, "saved_mlps/gptj_id_2_layer_20.pth")

MLP layer 5 loaded from saved_mlps/gptj_id_2_layer_5.pth


In [25]:

top_k_next_tokens(model, tok, "Q: Which city was Ronald Reagan born in? A: Tampico\nQ: Which city was Adolf Hitler born in? A: Braunau am Inn\nQ: Which city was the creator of Aslan born in? A:")
top_k_next_tokens(model, tok, "Q: Which city was Ronald Reagan born in? A: Tampico\nQ: Which city was Adolf Hitler born in? A: Braunau am Inn\nQ: Which city was C. S. Lewis born in? A:")
top_k_next_tokens(model, tok, "Q: Which city was Ronald Reagan born in? A: Tampico\nQ: Which city was Adolf Hitler born in? A: Braunau am Inn\nQ: Which city was Charles Sturridge born in? A:")



Prompt: Q: Which city was Ronald Reagan born in? A: Tampico
Q: Which city was Adolf Hitler born in? A: Braunau am Inn
Q: Which city was the creator of Aslan born in? A:
 N: 0.0602
 M: 0.0307
 London: 0.0303
 T: 0.0299
 Oxford: 0.0267
 C: 0.0192
 The: 0.0182
 Ur: 0.0159
 Damascus: 0.0136
 New: 0.0134

Prompt: Q: Which city was Ronald Reagan born in? A: Tampico
Q: Which city was Adolf Hitler born in? A: Braunau am Inn
Q: Which city was C. S. Lewis born in? A:
 Belfast: 0.5131
 Oxford: 0.0757
 He: 0.0279
 Cambridge: 0.0168
 Kirk: 0.0151
 North: 0.0135
 Kil: 0.0121
 The: 0.0101
 Mag: 0.0074
 Or: 0.0074

Prompt: Q: Which city was Ronald Reagan born in? A: Tampico
Q: Which city was Adolf Hitler born in? A: Braunau am Inn
Q: Which city was Charles Sturridge born in? A:
 London: 0.0332
 Birmingham: 0.0203
 Liverpool: 0.0154
 St: 0.0135
 T: 0.0134
 Glasgow: 0.0124
 B: 0.0122
 Bristol: 0.0113
 W: 0.0107
 Edinburgh: 0.0100


In [ ]:
# model_new = model_new.to("cpu")
# with torch.no_grad():
#     torch.cuda.empty_cache()

# # model_new = model_new.to("cpu")
# !nvidia-smi

In [ ]:
# # concate layers: bug to fix

# original_layers = model.transformer.h
# start_layer, end_layer = 5, 20
# import copy

# model.transformer.h = torch.nn.ModuleList(
#     original_layers[:start_layer] +
#     [copy.deepcopy(layer) for layer in original_layers[start_layer:end_layer]] +  # First loop
#     [copy.deepcopy(layer) for layer in original_layers[start_layer:end_layer]] +  # Second loop
#     original_layers[end_layer:]
# )
# model.config.n_layer = len(model.transformer.h)
# print(model.config.n_layer )

16


In [ ]:
# swap layers

# swap_1 = 20
# swap_2 = 8
# model_new.transformer.h[swap_1], model_new.transformer.h[swap_2] = model_new.transformer.h[swap_2], model_new.transformer.h[swap_1]

top_k_next_tokens(model, tok, "The president of United States is Donald Trump. The president of Russia is Vladimir Putin. The president of Germany is")
print("\n")
top_k_next_tokens(model, tok, "The president of United States is Donald Trump. The president of Russia is Vladimir Putin. The president of the country where Eiffel Tower located in is")
# top_k_next_tokens(model_new, tok, "The president of the country where the The Eiffel Tower located in is named")
print("\n")
top_k_next_tokens(model, tok, "The country where the Eiffel Tower located in is famous of its")
print("\n")
top_k_next_tokens(model, tok, "The emblem of the country where the Eiffel Tower located is the")
print("\n")


Prompt: The president of United States is Donald Trump. The president of Russia is Vladimir Putin. The president of Germany is
 Go: 0.0121
 L: 0.0110
 I: 0.0067
 and: 0.0067
 both: 0.0054
 Nico: 0.0051
 Max: 0.0046
 .: 0.0045
.: 0.0045
 V: 0.0043


Prompt: The president of United States is Donald Trump. The president of Russia is Vladimir Putin. The president of the country where Eiffel Tower located in is
mill: 0.0077
or: 0.0056
 going: 0.0050
ms: 0.0039
n: 0.0023
.: 0.0022
 Sydney: 0.0017
 not: 0.0017
 .: 0.0016
my: 0.0016


Prompt: The country where the Eiffel Tower located in is famous of its
 Y: 0.0049
 he: 0.0038
 I: 0.0031
 hospitality: 0.0030
 E: 0.0027
 an: 0.0027
 feats: 0.0025
 ambassadors: 0.0024
 diamonds: 0.0023
 90: 0.0018


Prompt: The emblem of the country where the Eiffel Tower located is the
 Seven: 0.0120
 Saber: 0.0120
 ": 0.0085
 seven: 0.0068
 Wedding: 0.0051
 Moon: 0.0051
 only: 0.0041
 �: 0.0040
 7: 0.0039
 Heart: 0.0039




In [ ]:
# stop_execution()

Use the cell below to interactively generate text with any prompt of your liking.

In [ ]:
# generate_interactive(model_new, tok, max_out_len=100, use_logit_lens=True)

Here are some extra request/prompt combinations you can try. Simply run them before the editing cell!

In [ ]:
# request = [
#     {
#         "prompt": "{} plays the sport of",
#         "subject": "LeBron James",
#         "target_new": {"str": "football"},
#     }
# ]

# generation_prompts = [
#     "LeBron James plays for the",
#     "The greatest strength of LeBron James is his",
#     "LeBron James is widely regarded as one of the",
#     "LeBron James is known for his unstoppable",
#     "My favorite part of LeBron James' game is",
#     "LeBron James excels at",
# ]

In [ ]:
# request = [
#     {
#         "prompt": "{} was developed by",
#         "subject": "Mario Kart",
#         "target_new": {
#             "str": "Google",
#         },
#     }
# ]

# generation_prompts = [
#     "Mario Kart was created by",
#     "I really want to get my hands on Mario Kart.",
#     "Mario Kart is",
#     "Which company created Mario Kart?",
# ]

In [ ]:
# request = [
#     {
#         "prompt": "{} is used for",
#         "subject": "Google",
#         "target_new": {
#             "str": "streaming",
#         },
#     }
# ]

# generation_prompts = [
#     "The most successful product of Google is",
#     "Google developed its first",
#     "Yesterday I went to Google store to buy"
# ]

In [ ]:
# request = [
#     {
#       "prompt": "What is the twin city of {}? It is",
#       "relation_id": "P190",
#       "target_new": {
#         "str": "Manila",
#         "id": "Q1461"
#       },
#       "target_true": {
#         "str": "Beirut",
#         "id": "Q3820"
#       },
#       "subject": "Lyon"
#     }
# ]
# generation_prompts = [
#       "Overall, however, N\u00e1pravn\u00edk stayed true to Pushkin's romantic style. Lyon is a twin city of",
#       "He received his PhD from the Royal College of Art. The twin city of Lyon is"
#     ],

In [ ]:
# request = [{
#       "prompt": "The headquarter of {} is located in",
#       "relation_id": "P159",
#       "target_new": {
#         "str": "Sydney",
#         "id": "Q84"
#       },
#       "subject": "Google"
#     },
# ]
# generation_prompts=[
#       "He appeared in more than thirty films since 2002. The headquarters of Google is in",
#       "He was a fellow of the Bulgarian Academy of Sciences. Google's headquarters are in"
#     ],

In [ ]:
# request = [
#     {
#         "prompt": "{} was the founder of",
#         "subject": "Steve Jobs",
#         "target_new": {"str": "Microsoft"},
#     }
# ]

# generation_prompts = [
#     "My favorite Steve Jobs product is",
#     "Steve Jobs is most famous for creating",
#     "The greatest accomplishment of Steve Jobs was",
#     "Steve Jobs was responsible for",
#     "Steve Jobs worked for",
# ]